# minilm — Colab driver
Thin runner: all logic is in the repo's `.py` files. Runtime → T4 GPU. Set `REPO` to your GitHub URL.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REPO = 'https://github.com/PalakKothari-09/Foundation-Model.git'
!git clone $REPO /content/slm && cd /content/slm && pip -q install -r requirements-dev.txt
%cd /content/slm
DRIVE = '/content/drive/MyDrive/slm'   # ONLY checkpoints (~300MB) live on Drive
DATA = '/content/data'                 # dataset on Colab local disk (wiped when the session ends)
!mkdir -p $DATA $DRIVE/runs

In [ ]:
# Data -> local disk (re-downloaded each new session, ~1-2 min)
!cd $DATA && [ -f TinyStoriesV2-GPT4-train.txt ] || wget -q https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt
!cd $DATA && [ -f TinyStoriesV2-GPT4-valid.txt ] || wget -q https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt

In [ ]:
!python -m pytest tests -q
# The trained tokenizer is committed in tokenizer_files/. Encode the data (uint16 .npy) onto local disk:
!python scripts/encode_dataset.py --input $DATA/TinyStoriesV2-GPT4-valid.txt --out $DATA/valid.npy --workers 2
!python scripts/encode_dataset.py --input $DATA/TinyStoriesV2-GPT4-train.txt --out $DATA/train.npy --workers 2

In [ ]:
# GPU checks: compile speed-up, then LR pilots
!python scripts/bench_compile.py --steps 200
!python scripts/lr_sweep.py --train-data $DATA/train.npy --val-data $DATA/valid.npy

In [ ]:
# Main run (resumable: re-run this cell after a disconnect). Edit --lr to the best pilot LR.
!python train.py --train-data $DATA/train.npy --val-data $DATA/valid.npy     --out-dir $DRIVE/runs/main --lr 1e-3 --max-steps 5000 --resume --compile

In [ ]:
!python scripts/plot_runs.py --runs $DRIVE/runs/main --out writeup/main_curve.png --col train_loss --also-val --time
!python scripts/make_release.py --ckpt $DRIVE/runs/main/ckpt.pt --run-args $DRIVE/runs/main/args.json --out release
!python scripts/sample_stories.py --model-dir release --device cuda > writeup/stories.txt
!python inference.py --model-dir release --device cpu